# MLB injured-list stints, 2015–2019, from MLB transactions (keyed by MLBAM ID)

Builds an injured-list table for seasons before the FanGraphs injury workbook begins, with the **same
columns as the FanGraphs sheets**

**Source.** The MLB Stats API transactions endpoint
(`https://statsapi.mlb.com/api/v1/transactions?sportId=1&startDate=…&endDate=…`), which is what the
mlb.com transactions pages are built on. Each transaction carries the player's **MLBAM ID** (`person.id`,
the same number as in his mlb.com player link), the date, the teams and the text, e.g.
*"Los Angeles Dodgers placed LHP Clayton Kershaw on the 15-day disabled list retroactive to June 27, 2016.
Herniated disc."* Using the API avoids scraping the HTML pages.

**Steps**

1. Download transactions month by month (cached as JSON in `data/mlb_transactions/`).
2. Classify injured-list transactions from their text: **placed**, **transferred** (e.g. 15-day → 60-day),
   **activated / reinstated**; plus **released / elected free agency / retired**, which end a stint without a
   return. Both wordings are handled: "disabled list" (through 2018) and "injured list" (2019 on).
3. Build stints per MLBAM ID: placement → activation. The **IL date** is the retroactive date when given,
   otherwise the placement date (FanGraphs' "IL Retro Date"); the **return date** is the activation date.
4. **Carry-overs across seasons.** A stint still open at the end of a season gets a row in that season with
   no return date, and a row in the next season (IL date = that season's re-placement date if MLB issued
   one, otherwise opening day), with **"Injury / Surgery Date" = the original IL date**. The v4/v5 baseline
   window then ends at the original injury, as it does for FanGraphs carry-overs.
5. Add `key_fangraphs` from the Chadwick register (the v4/v5 pipeline keys pitchers by FanGraphs ID) and keep
   `key_mlbam` as the primary key.
6. **Validate** by running the same steps on 2021–2023 and comparing with the FanGraphs workbook.
7. Write a combined workbook: the FanGraphs sheets unchanged plus one sheet per new season.

## 1. Settings

In [4]:
#%pip install numpy
%pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 33.4 MB/s  0:00:006m0:00:01

[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [9]:
from pathlib import Path
import json
import re
import time
from urllib.request import Request, urlopen

import numpy as np
import pandas as pd
from IPython.display import display

WORKSPACE = Path.cwd()
DATA_DIR = WORKSPACE / "data"
TX_CACHE = DATA_DIR / "mlb_transactions"
TX_CACHE.mkdir(parents=True, exist_ok=True)
CHADWICK_CACHE = DATA_DIR / "chadwick_register_ids.csv"          # shared with the v4 / v5 notebooks
FANGRAPHS_WORKBOOK = Path("/workspaces/Sports-Analytics-and-Intelligence-Lab/MLB Oblique Injuries/MLB Injuries Raw Data.xlsx")     # 2021-2026, FanGraphs
OUTPUT_WORKBOOK = WORKSPACE / "MLB Injuries Raw Data 2015-2026.xlsx"

BUILD_SEASONS = [2015, 2016, 2017, 2018, 2019]
VALIDATE_SEASONS = [2021, 2022, 2023]     # rebuilt from MLB transactions and compared with FanGraphs ([] to skip)
FETCH_RANGES = [("2014-01-01", "2019-12-31")]                     # 2014 so 2015 carry-overs are found
if VALIDATE_SEASONS:
    FETCH_RANGES.append((f"{min(VALIDATE_SEASONS) - 1}-01-01", f"{max(VALIDATE_SEASONS)}-12-31"))

TX_URL = "https://statsapi.mlb.com/api/v1/transactions?sportId=1&startDate={start}&endDate={end}"

# Opening days (domestic), used only for the IL date of carry-over rows with no spring re-placement.
OPENING_DAY = {2014: "2014-03-31", 2015: "2015-04-05", 2016: "2016-04-03", 2017: "2017-04-02",
               2018: "2018-03-29", 2019: "2019-03-28", 2020: "2020-07-23", 2021: "2021-04-01",
               2022: "2022-04-07", 2023: "2023-03-30", 2024: "2024-03-28", 2025: "2025-03-27"}
OPENING_DAY = {y: pd.Timestamp(d) for y, d in OPENING_DAY.items()}
# Last day of the regular season (incl. tiebreakers / make-ups). Activations after it, before the next
# Opening Day, are off-season reinstatements (all 60-day IL players are reinstated after the World Series),
# not returns to play: the stint ends with no return date, as in FanGraphs.
SEASON_END = {2014: "2014-09-28", 2015: "2015-10-04", 2016: "2016-10-02", 2017: "2017-10-01",
              2018: "2018-10-01", 2019: "2019-09-29", 2020: "2020-09-27", 2021: "2021-10-03",
              2022: "2022-10-05", 2023: "2023-10-01", 2024: "2024-09-30", 2025: "2025-09-28"}
SEASON_END = {y: pd.Timestamp(d) for y, d in SEASON_END.items()}
SPRING_BACKDATE = pd.Timedelta(days=3)   # a pre-season IL placement counts from Opening Day minus 3 days

def in_offseason(d):
    """True if date d falls after a regular season ended and before the next Opening Day."""
    end = SEASON_END.get(d.year, pd.Timestamp(f"{d.year}-10-05"))
    start = OPENING_DAY.get(d.year, pd.Timestamp(f"{d.year}-03-28"))
    return d > end or d < start
PITCHER_TOKENS = {"RHP", "LHP", "P"}
DUP_WINDOW_DAYS = 7          # a repeat of the same transaction within 7 days is treated as a duplicate listing

## 2. Download transactions (cached by month)

One request per calendar month. Files already in `data/mlb_transactions/` are reused, so re-running only
downloads what is missing. The cell then prints the fields of a few raw records to confirm the structure.

In [ ]:
def month_starts(start, end):
    return pd.date_range(pd.Timestamp(start).replace(day=1), pd.Timestamp(end), freq="MS")


def fetch_month(m, retries=3, sleep_seconds=0.5):
    path = TX_CACHE / f"transactions_{m:%Y_%m}.json"
    if path.exists():
        return json.loads(path.read_text())
    end = (m + pd.offsets.MonthEnd(0))
    url = TX_URL.format(start=f"{m:%Y-%m-%d}", end=f"{end:%Y-%m-%d}")
    for attempt in range(1, retries + 1):
        try:
            time.sleep(sleep_seconds)
            with urlopen(Request(url, headers={"User-Agent": "Mozilla/5.0"}), timeout=60) as r:
                payload = json.loads(r.read().decode("utf-8"))
            path.write_text(json.dumps(payload))
            return payload
        except Exception as error:
            print(f"  {m:%Y-%m} attempt {attempt} failed: {error!r}")
            time.sleep(5 * attempt)
    raise RuntimeError(f"Could not download transactions for {m:%Y-%m}")


raw = []
for start, end in FETCH_RANGES:
    for m in month_starts(start, end):
        raw.extend(fetch_month(m).get("transactions", []))
print(f"Transactions downloaded: {len(raw):,}")
keys = pd.Series([k for t in raw[:2000] for k in t]).value_counts()
print("Fields present (first 2,000 records):", dict(keys))
for t in [t for t in raw if "list" in str(t.get("description", "")).lower()][:3]:
    print(json.dumps(t, indent=1)[:900])

Transactions downloaded: 138,670
Fields present (first 2,000 records): {'id': np.int64(2000), 'toTeam': np.int64(2000), 'date': np.int64(2000), 'effectiveDate': np.int64(2000), 'typeCode': np.int64(2000), 'typeDesc': np.int64(2000), 'description': np.int64(2000), 'person': np.int64(1992), 'resolutionDate': np.int64(1688), 'fromTeam': np.int64(315)}
{
 "id": 177836,
 "person": {
  "id": 121347,
  "fullName": "Alex Rodriguez",
  "link": "/api/v1/people/121347",
  "nameSlug": "alex-rodriguez-121347"
 },
 "fromTeam": {
  "id": 147,
  "name": "New York Yankees",
  "link": "/api/v1/teams/147"
 },
 "toTeam": {
  "id": 147,
  "name": "New York Yankees",
  "link": "/api/v1/teams/147"
 },
 "date": "2014-01-11",
 "effectiveDate": "2014-01-11",
 "resolutionDate": "2014-01-11",
 "typeCode": "SC",
 "typeDesc": "Status Change",
 "description": "New York Yankees placed 3B Alex Rodriguez on the restricted list."
}
{
 "id": 178758,
 "person": {
  "id": 455092,
  "fullName": "Scott Elbert",
  "link": "/a

## 3. Flatten and classify injured-list transactions

The MLBAM ID comes from `person.id`. The text decides the event type. Descriptions that mention the
injured/disabled list but match none of the patterns are listed for review at the end of the cell (rehab
assignments and similar notes are expected there).

In [15]:
def team_name(t):
    for k in ("toTeam", "fromTeam"):
        if isinstance(t.get(k), dict) and t[k].get("name"):
            return t[k]["name"]
    return None


tx = pd.DataFrame([{
    "tx_id": t.get("id"), "mlbam": (t.get("person") or {}).get("id"),
    "name": (t.get("person") or {}).get("fullName"), "team": team_name(t),
    "date": t.get("date"), "effective_date": t.get("effectiveDate"), "resolution_date": t.get("resolutionDate"),
    "type_code": t.get("typeCode"), "type_desc": t.get("typeDesc"),
    "description": (t.get("description") or "").strip()} for t in raw])
tx = tx.dropna(subset=["mlbam"]).drop_duplicates("tx_id" if tx["tx_id"].notna().all() else None)
tx["mlbam"] = tx["mlbam"].astype(int)
for c in ["date", "effective_date", "resolution_date"]:
    tx[c] = pd.to_datetime(tx[c], errors="coerce")
tx["event_date"] = tx["date"].fillna(tx["effective_date"])

# ---- Layer 1: exact repeats of the same transaction ----
tx["norm_desc"] = (tx["description"].str.lower().str.replace(r"\s+", " ", regex=True)
                   .str.strip().str.rstrip(".").str.strip())
tx = tx.sort_values(["mlbam", "norm_desc", "event_date", "tx_id"]).reset_index(drop=True)
gap = tx.groupby(["mlbam", "norm_desc"])["event_date"].diff().dt.days
tx["duplicate"] = gap.notna() & (gap <= DUP_WINDOW_DAYS)
tx["duplicate_of_id"] = tx.groupby(["mlbam", "norm_desc"])["tx_id"].shift().where(tx["duplicate"])
dups = tx.loc[tx["duplicate"]]
dups.to_csv(DATA_DIR / "mlb_transactions_duplicates_removed.csv", index=False)
print(f"Exact duplicate listings removed: {len(dups):,} of {len(tx):,} transactions "
      f"({dups['mlbam'].nunique():,} players); same-day repeats: {int((gap[tx['duplicate']] == 0).sum()):,}")
tx = tx.loc[~tx["duplicate"]].drop(columns=["duplicate", "duplicate_of_id"]).reset_index(drop=True)

POS = r"(?P<pos>RHP|LHP|P|C|1B|2B|3B|SS|LF|CF|RF|OF|IF|INF|DH|UT)"
LIST = r"(?P<days>\d+)[- ]day (?:disabled list|injured list|DL|IL)"   # "10-day" and "10 day" both occur
DATE_TXT = r"(?P<retro>[A-Z][a-z]{2,8}\.? \d{1,2}(?:, \d{4})?)"    # year sometimes omitted
RX = {
    "placed": re.compile(rf"\bplaced {POS} (?P<name>.+?) on the {LIST}(?: retroactive to {DATE_TXT})?[.,;]?\s*(?P<injury>.*)$"),
    "transferred": re.compile(rf"\btransferred {POS} (?P<name>.+?) (?:from the \d+[- ]day (?:disabled list|injured list|DL|IL) )?"
                              rf"to the {LIST}(?: retroactive to {DATE_TXT})?[.,;]?\s*(?P<injury>.*)$"),
    "activated": re.compile(rf"\b(?:activated|reinstated) {POS} (?P<name>.+?) from the {LIST}"),
    # Bare "activated RHP X." (no list named; common from 2021). Counts as a return only if a stint is open.
    "activated_bare": re.compile(rf"\b(?:activated|reinstated) {POS} (?P<name>(?:(?!\bfrom the\b).)+?)\.*\s*$"),
    # Leaving the active roster while on the IL implies he was activated that day (FanGraphs uses it as the return).
    "off_roster": re.compile(r"\b(?:optioned |designated .+? for assignment|sent .+? outright\b)"),
}
ENDERS = re.compile(r"\b(released|elected free agency|retired|became a free agent)\b", re.I)
LIST_WORDS = re.compile(r"(?:disabled|injured) list|\b\d+[- ]day (?:DL|IL)\b")


def parse_date_text(s, event_date=None):
    """'June 27, 2016' / 'Sept. 1, 2019'; without a year, the most recent such date on or before the event."""
    if not isinstance(s, str):
        return pd.NaT
    s = s.replace(".", "").replace("Sept ", "Sep ").strip()
    if not re.search(r"\d{4}$", s):
        if event_date is None or pd.isna(event_date):
            return pd.NaT
        d = parse_date_text(f"{s}, {event_date.year}")
        return d - pd.DateOffset(years=1) if pd.notna(d) and d > event_date else d
    for fmt in ("%B %d, %Y", "%b %d, %Y"):
        try:
            return pd.to_datetime(s, format=fmt)
        except ValueError:
            pass
    return pd.NaT


def classify(desc):
    for kind, rx in RX.items():
        m = rx.search(desc)
        if m:
            g = m.groupdict()
            injury = (g.get("injury") or "").strip().rstrip(".").strip()
            days = int(g["days"]) if g.get("days") else None
            return kind, g.get("pos"), days, g.get("retro"), injury
    if ENDERS.search(desc):
        return "ended", None, None, pd.NaT, ""
    return None, None, None, pd.NaT, ""


parsed = tx["description"].map(classify)
tx["event"] = parsed.map(lambda p: p[0])
tx["pos"] = parsed.map(lambda p: p[1])
tx["list_days"] = parsed.map(lambda p: p[2])
tx["retro_date"] = [parse_date_text(r, d) for r, d in zip(parsed.map(lambda p: p[3]), tx["event_date"])]
# A retro date with an obvious typo in the year (e.g. "April 8, 2022" on an April 2023 placement): drop the year
bad_year = (tx["event_date"] - tx["retro_date"]).dt.days.abs() > 60
tx.loc[bad_year, "retro_date"] = [parse_date_text(re.sub(r",\s*\d{4}$", "", r), d)
                                  for r, d in zip(parsed[bad_year].map(lambda p: p[3]), tx.loc[bad_year, "event_date"])]
print(f"Retro dates with a mistyped year, corrected: {int(bad_year.sum())}")
tx["injury"] = parsed.map(lambda p: p[4])
events = tx.loc[tx["event"].notna()].sort_values(["mlbam", "event_date", "tx_id"]).reset_index(drop=True)

display(events["event"].value_counts().rename("transactions"))
unmatched = tx.loc[tx["event"].isna() & tx["description"].str.contains(LIST_WORDS)]
unmatched.to_csv(DATA_DIR / "mlb_transactions_unmatched_IL_text.csv", index=False)
print(f"IL-related descriptions not classified: {len(unmatched)} (saved for review). Most common openings:")
display(unmatched["description"].str.replace(r"^[A-Z][\w.' ]+? (?=[a-z])", "", regex=True)
        .str.split().str[:4].str.join(" ").value_counts().head(15))

Exact duplicate listings removed: 4,910 of 129,840 transactions (2,376 players); same-day repeats: 3,702
Retro dates with a mistyped year, corrected: 16


event
off_roster        20902
placed             7581
activated          5531
ended              3921
activated_bare     3771
transferred        1211
Name: transactions, dtype: int64

IL-related descriptions not classified: 2 (saved for review). Most common openings:


description
placed SHP Pat Venditte         1
transferred SHP Pat Venditte    1
Name: count, dtype: int64

## 4. Build stints per player (MLBAM ID)

Events are processed in date order for each player:

* **placed** opens a stint (IL date = retroactive date if given). If he is already on the IL, it is a
  re-placement: same stint, and if it falls in a new season it starts that season's row (spring
  re-placements of players who ended the previous season on the IL).
* **transferred** keeps the stint open (opens one if the placement was missed, flagged).
* **activated** closes it; return date = activation date.
* **released / free agency / retired** closes it with no return date.
* Open at the end of the data → no return date.

A stint spanning seasons is written as one row per season (see the introduction).

In [16]:
def emit_rows(st, ret, status):
    """Split one stint into per-season rows."""
    rows = []
    years = sorted(set(range(st["il_date"].year, (ret.year if pd.notna(ret) else st["last_year"]) + 1)))
    for y in years:
        first = y == st["il_date"].year
        last = y == years[-1]
        od = OPENING_DAY.get(y, pd.Timestamp(f"{y}-04-01"))
        il = st["il_date"] if first else st["season_start"].get(y, od - SPRING_BACKDATE)
        il = max(il, od - SPRING_BACKDATE)   # pre-season placements: Opening Day minus 3 days, as FanGraphs/MLB date them
        rows.append({
            "mlbam": st["mlbam"], "Name": st["name"], "Team": st["team"], "pos_token": st["pos"],
            "Season": y, "IL Retro Date": il, "placement_date": st["placed_on"] if first else st["season_start"].get(y, pd.NaT),
            "Injury / Surgery Date": pd.NaT if first else st["il_date"],
            "Injury / Surgery": st["injury_by_year"].get(y) or st["injury"],
            "Return Date": ret if last else pd.NaT,
            "Status": status if last else "Season ended on IL",
            "list_days_max": st["days_max"], "opened_by": st["opened_by"]})
    return rows


rows = []
flags = {"activation without open stint": 0, "stint opened by transfer": 0, "re-placement while on IL (new season)": 0,
         "duplicate: placement while on IL (same season, within window)": 0,
         "duplicate: activation (no open stint, within window of last activation)": 0,
         "duplicate: placement re-posted after activation": 0, "activation in off-season (no return that season)": 0,
         "stint closed by bare activation": 0, "stint closed by option / DFA / outright": 0}
WINDOW = pd.Timedelta(days=DUP_WINDOW_DAYS)
for mlbam, ev in events.groupby("mlbam", sort=False):
    st, last = None, None            # last = the most recently closed stint: return date, injury, list days
    for _, e in ev.iterrows():
        d = e["event_date"]
        il_e = e["retro_date"] if pd.notna(e["retro_date"]) else d
        if e["event"] == "placed" and st is None and last is not None and pd.notna(last["ret"]) and il_e < last["ret"]:
            flags["duplicate: placement re-posted after activation"] += 1
            continue
        if e["event"] == "placed" or (e["event"] == "transferred" and st is None):
            if st is None:
                il = e["retro_date"] if pd.notna(e["retro_date"]) else d
                st = {"mlbam": mlbam, "name": e["name"], "team": e["team"], "pos": e["pos"], "il_date": il,
                      "placed_on": d, "injury": e["injury"], "injury_by_year": {il.year: e["injury"]},
                      "days_max": e["list_days"], "season_start": {}, "last_year": il.year,
                      "opened_by": e["event"]}
                if e["event"] == "transferred":
                    flags["stint opened by transfer"] += 1
            else:
                if d.year == st["last_year"] and d - st["placed_on"] <= WINDOW:
                    flags["duplicate: placement while on IL (same season, within window)"] += 1
                    st["injury"] = st["injury"] or e["injury"]
                    continue
                if d.year > st["last_year"]:
                    flags["re-placement while on IL (new season)"] += 1
                    st["season_start"][d.year] = e["retro_date"] if pd.notna(e["retro_date"]) else d
                    if e["injury"]:
                        st["injury_by_year"][d.year] = e["injury"]
                st["days_max"] = max(st["days_max"] or 0, e["list_days"] or 0)
                st["injury"] = st["injury"] or e["injury"]
                st["team"] = e["team"] or st["team"]
            st["last_year"] = max(st["last_year"], d.year)
        elif e["event"] == "transferred":
            st["days_max"] = max(st["days_max"] or 0, e["list_days"] or 0)
            st["injury"] = st["injury"] or e["injury"]
            st["last_year"] = max(st["last_year"], d.year)
        elif e["event"] in ("activated_bare", "off_roster") and st is None:
            continue                                   # routine move, no open stint: nothing to do
        elif e["event"] in ("activated", "activated_bare", "off_roster"):
            if st is None:
                if last is not None and pd.notna(last["ret"]) and d - last["ret"] <= WINDOW:
                    flags["duplicate: activation (no open stint, within window of last activation)"] += 1
                else:
                    flags["activation without open stint"] += 1
                continue
            if in_offseason(d):
                flags["activation in off-season (no return that season)"] += 1
                rows += emit_rows(st, pd.NaT, "Season ended on IL (reinstated in off-season)")
                last = None
            else:
                status = {"activated": "Activated", "activated_bare": "Activated",
                          "off_roster": "Activated and optioned / designated / outrighted"}[e["event"]]
                if e["event"] != "activated":
                    flags[f"stint closed by {'bare activation' if e['event'] == 'activated_bare' else 'option / DFA / outright'}"] += 1
                rows += emit_rows(st, d, status)
                last = {"ret": d, "injury": st["injury"], "days": st["days_max"]}
            st = None
        elif e["event"] == "ended" and st is not None:
            rows += emit_rows(st, pd.NaT, "Released / free agent / retired while on IL")
            last, st = None, None
    if st is not None:
        rows += emit_rows(st, pd.NaT, "On IL at end of data")

stints = pd.DataFrame(rows)
stints["Pos"] = np.where(stints["pos_token"].isin(PITCHER_TOKENS), "P", stints["pos_token"])
display(pd.Series(flags, name="count").to_frame())
display(stints.groupby(["Season", "Pos"]).size().unstack(fill_value=0).loc[:, lambda d: d.columns.isin(["P", "C", "OF", "SS"])])
display(stints.loc[stints["Pos"] == "P"].groupby("Season")["Status"].value_counts().unstack(fill_value=0))

,count
activation without open stint,125
stint opened by transfer,37
re-placement while on IL (new season),26
"duplicate: placement while on IL (same season, within window)",135
"duplicate: activation (no open stint, within window of last activation)",144
duplicate: placement re-posted after activation,2
activation in off-season (no return that season),1694
stint closed by bare activation,644
stint closed by option / DFA / outright,672


Pos,C,OF,P,SS
Season,,,,
2014,31,0,238,22
2015,35,0,288,16
2016,46,0,302,23
2017,47,3,383,30
2018,35,3,432,29
2019,52,1,400,27
2020,33,7,317,21
2021,71,9,689,77
2022,65,15,563,65


Status,Activated,Activated and optioned / designated / outrighted,On IL at end of data,Released / free agent / retired while on IL,Season ended on IL,Season ended on IL (reinstated in off-season)
Season,,,,,,
2014,123,33,0,7,5,70
2015,142,39,1,11,8,87
2016,165,32,1,5,11,88
2017,220,54,0,18,10,81
2018,252,68,0,6,13,93
2019,234,58,1,16,9,82
2020,139,33,0,10,7,128
2021,454,41,0,35,7,152
2022,336,33,0,17,14,163


## 5. FanGraphs IDs and the FanGraphs-style table

`key_mlbam` stays the primary key. `key_fangraphs` is added from the Chadwick register because the v4/v5
pipeline keys pitchers by FanGraphs ID; pitchers without one are listed.

In [17]:
def clean_id(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    return (s[:-2] if s.endswith(".0") else s) or None


if CHADWICK_CACHE.exists():
    id_map = pd.read_csv(CHADWICK_CACHE, dtype=str)
else:
    url = "https://raw.githubusercontent.com/chadwickbureau/register/master/data/people-{}.csv"
    id_map = pd.concat([pd.read_csv(url.format(s), usecols=["key_mlbam", "key_fangraphs"], dtype=str).dropna()
                        for s in "0123456789abcdef"], ignore_index=True)
    id_map.to_csv(CHADWICK_CACHE, index=False)
mlbam_to_fg = dict(zip(id_map["key_mlbam"].map(clean_id), id_map["key_fangraphs"].map(clean_id)))

out = stints.copy()
out["key_mlbam"] = out["mlbam"].astype(str)
out["key_fangraphs"] = out["key_mlbam"].map(mlbam_to_fg)
out["URL"] = "https://www.mlb.com/player/" + out["key_mlbam"]
out["Eligible to Return"] = pd.NaT
out["Latest Update"] = ""
out["source"] = "MLB Stats API transactions"
COLS = ["Name", "URL", "Team", "Pos", "Injury / Surgery Date", "Injury / Surgery", "Status", "IL Retro Date",
        "Eligible to Return", "Return Date", "Latest Update", "key_fangraphs", "key_mlbam",
        "placement_date", "list_days_max", "opened_by", "source"]
out = out[COLS + ["Season"]].sort_values(["Season", "IL Retro Date", "Name"]).reset_index(drop=True)
no_fg = out.loc[(out["Pos"] == "P") & out["key_fangraphs"].isna()]
print(f"Pitcher rows without a FanGraphs ID: {len(no_fg)} of {(out['Pos'] == 'P').sum()}")
display(no_fg[["Season", "Name", "key_mlbam"]].head(20))

Pitcher rows without a FanGraphs ID: 18 of 4109


,Season,Name,key_mlbam
950,2015,Grayson Garvin,543204
1085,2016,Zack Jones,572958
1531,2016,Chris Lee,595296
1554,2017,Armando Rivero,633831
1612,2017,Kevin Gadea,640737
1679,2017,Nick Travieso,621072
2213,2017,Dylan Baker,621173
2261,2018,Clayton Blackburn,607185
2336,2018,Rayan González,543234
3076,2019,Riley Ferrell,621068


## 6. Validation against FanGraphs (2021–2023)

The same steps, applied to 2021–2023 transactions, should reproduce the FanGraphs workbook's pitcher rows.
A FanGraphs row counts as matched when the same pitcher (MLBAM ID) has a rebuilt row in that season with an
IL date within 3 days. Return dates are then compared for matched rows. The unmatched lists are saved so a
pattern (e.g. a wording the parser misses) can be spotted.

In [17]:
%pip install openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]1/2 [openpyxl]

[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [18]:
if VALIDATE_SEASONS and FANGRAPHS_WORKBOOK.exists():
    # ==== Section 6 (replacement): validation against FanGraphs, with a reason for every unmatched row ====
    # A FanGraphs row and a rebuilt row are the same pitcher if EITHER id agrees (MLBAM via Chadwick, or the
    # FanGraphs id directly), so one bad link in the ID register can no longer make a correct row look unmatched.
    # Each unmatched row now shows the nearest row for the same pitcher in the other source and why it failed.

    fg_to_mlbams = (id_map.assign(fg=id_map["key_fangraphs"].map(clean_id), mb=id_map["key_mlbam"].map(clean_id))
                    .dropna(subset=["fg", "mb"]).groupby("fg")["mb"].agg(lambda s: sorted(set(s))).to_dict())
    TOL = 3


    def link_pairs(fg, mine):
        """All (FanGraphs row, rebuilt row) pairs that share a pitcher by either id, with the IL-date gap."""
        fg_long = fg[["rid", "fg_id"]].assign(mb=fg["fg_id"].map(lambda v: fg_to_mlbams.get(v, []))).explode("mb")
        by_mb = fg_long.dropna(subset=["mb"]).merge(mine[["tid", "key_mlbam"]], left_on="mb", right_on="key_mlbam")
        by_fg = fg[["rid", "fg_id"]].dropna(subset=["fg_id"]).merge(mine[["tid", "key_fangraphs"]].dropna(),
                                                                   left_on="fg_id", right_on="key_fangraphs")
        p = pd.concat([by_mb[["rid", "tid"]].assign(link="mlbam"), by_fg[["rid", "tid"]].assign(link="fangraphs id")])
        p = p.groupby(["rid", "tid"])["link"].agg(lambda s: " + ".join(sorted(set(s)))).reset_index()
        p = p.merge(fg[["rid", "il", "ret"]], on="rid").merge(mine[["tid", "il", "ret"]], on="tid", suffixes=("_fg", "_tx"))
        p["gap"] = (p["il_tx"] - p["il_fg"]).dt.days.abs()
        return p


    def carry_note(r):
        if pd.notna(r.get("Injury / Surgery Date")) and pd.isna(r.get("placement_date")):
            return f"carry-over row: stint opened {r['Injury / Surgery Date']:%Y-%m-%d}, no re-placement found this season"
        if pd.notna(r.get("Injury / Surgery Date")):
            return f"carry-over row: stint opened {r['Injury / Surgery Date']:%Y-%m-%d}"
        return ""


    val_rows, rev = [], []
    for s in VALIDATE_SEASONS:
        fg = pd.read_excel(FANGRAPHS_WORKBOOK, sheet_name=str(s))
        fg = fg.loc[fg["Pos"].astype(str).str.strip().str.upper().isin({"RP", "SP", "SP/RP", "RP/SP", "P"})].copy()
        fg["fg_id"] = fg["key_fangraphs"].map(clean_id)
        fg["il"] = pd.to_datetime(fg["IL Retro Date"], errors="coerce")
        fg["ret"] = pd.to_datetime(fg["Return Date"], errors="coerce")
        fg = fg.dropna(subset=["il"]).reset_index(drop=True)
        fg["rid"] = fg.index
        mine = out.loc[(out["Season"] == s) & (out["Pos"] == "P")].copy().reset_index(drop=True)
        mine["tid"] = mine.index
        mine["il"] = pd.to_datetime(mine["IL Retro Date"])
        mine["ret"] = pd.to_datetime(mine["Return Date"])

        p = link_pairs(fg, mine)
        # one-to-one matching, closest IL dates first
        used_r, used_t, match = set(), set(), []
        for _, q in p.loc[p["gap"] <= TOL].sort_values("gap").iterrows():
            if q["rid"] not in used_r and q["tid"] not in used_t:
                used_r.add(q["rid"]); used_t.add(q["tid"]); match.append(q)
        M = pd.DataFrame(match, columns=p.columns)
        both = M.dropna(subset=["ret_fg", "ret_tx"])
        val_rows.append({"season": s, "FanGraphs pitcher rows": len(fg), "rebuilt pitcher rows": len(mine),
                         "matched (IL within 3 days)": f"{len(M) / len(fg):.1%}",
                         "linked by MLBAM only / FanGraphs id only": f"{(M['link'] == 'mlbam').sum()} / {(M['link'] == 'fangraphs id').sum()}",
                         "return date exact": f"{(both['ret_fg'] == both['ret_tx']).mean():.1%}",
                         "return within 3 days": f"{((both['ret_fg'] - both['ret_tx']).dt.days.abs() <= 3).mean():.1%}",
                         "return missing in one source only": int((M["ret_fg"].isna() != M["ret_tx"].isna()).sum())})

        linked_r, linked_t = set(p["rid"]), set(p["tid"])
        for _, r in fg.loc[~fg["rid"].isin(used_r)].iterrows():
            cand = p.loc[(p["rid"] == r["rid"])].sort_values("gap")
            if r["rid"] not in linked_r:
                why = ("no FanGraphs id in workbook" if r["fg_id"] is None else
                       "FanGraphs id not in Chadwick and no rebuilt row with that id" if r["fg_id"] not in fg_to_mlbams else
                       "pitcher has no rebuilt row this season")
                near = None
            else:
                near = mine.loc[cand["tid"].iloc[0]]
                why = (f"nearest rebuilt row {int(cand['gap'].iloc[0])} days off" if cand["gap"].iloc[0] > TOL
                       else "rebuilt row already matched to another FanGraphs row (FanGraphs lists this stint twice?)")
            rev.append({"season": s, "only in": "FanGraphs", "Name": r["Name"], "key_fangraphs": r["fg_id"],
                        "key_mlbam": near["key_mlbam"] if near is not None else ", ".join(fg_to_mlbams.get(r["fg_id"], [])),
                        "IL date": r["il"], "return": r["ret"], "injury": r["Injury / Surgery"],
                        "other source IL date": near["il"] if near is not None else pd.NaT,
                        "other source return": near["ret"] if near is not None else pd.NaT,
                        "reason": why, "note": carry_note(near) if near is not None else ""})
        for _, r in mine.loc[~mine["tid"].isin(used_t)].iterrows():
            cand = p.loc[(p["tid"] == r["tid"])].sort_values("gap")
            if r["tid"] not in linked_t:
                near, why = None, "no FanGraphs row for this pitcher this season"
            else:
                near = fg.loc[cand["rid"].iloc[0]]
                why = (f"nearest FanGraphs row {int(cand['gap'].iloc[0])} days off" if cand["gap"].iloc[0] > TOL
                       else "FanGraphs row already matched to another rebuilt row (stint split in two here?)")
            rev.append({"season": s, "only in": "rebuilt", "Name": r["Name"], "key_fangraphs": r["key_fangraphs"],
                        "key_mlbam": r["key_mlbam"], "IL date": r["il"], "return": r["ret"], "injury": r["Injury / Surgery"],
                        "other source IL date": near["il"] if near is not None else pd.NaT,
                        "other source return": near["ret"] if near is not None else pd.NaT,
                        "reason": why, "note": carry_note(r)})

    validation = pd.DataFrame(val_rows)
    review = pd.DataFrame(rev).sort_values(["season", "Name", "IL date"])
    review.to_csv(DATA_DIR / "validation_review.csv", index=False)
    display(validation)
    review["reason group"] = review["reason"].str.replace(r"\d+ days off", "N days off", regex=True)
    display(review.groupby(["only in", "reason group"]).size().rename("rows").to_frame())
    print("Every unmatched row, with its reason: data/validation_review.csv")


    # ==== Look up one pitcher: raw transactions, parsed events, rebuilt rows, FanGraphs rows ====
    def show_pitcher(name, season=None):
        ids = tx.loc[tx["name"].str.contains(name, case=False, na=False), "mlbam"].unique()
        for m in ids:
            print(f"\n=== {tx.loc[tx['mlbam'] == m, 'name'].iloc[0]}  (MLBAM {m}, FanGraphs {mlbam_to_fg.get(str(m))}) ===")
            t = tx.loc[tx["mlbam"] == m]
            if season:
                t = t.loc[t["event_date"].dt.year.between(season - 1, season)]
            with pd.option_context("display.max_colwidth", 140, "display.width", 250):
                display(t[["tx_id", "event_date", "event", "retro_date", "description"]])
                r = out.loc[out["key_mlbam"] == str(m)]
                display(r.loc[r["Season"] == season] if season else r)
            if season and FANGRAPHS_WORKBOOK.exists() and season in VALIDATE_SEASONS:
                f = pd.read_excel(FANGRAPHS_WORKBOOK, sheet_name=str(season))
                display(f.loc[f["Name"].str.contains(name, case=False, na=False),
                              ["Name", "IL Retro Date", "Return Date", "Injury / Surgery", "key_fangraphs"]])



    # ==== Every transaction for the pitchers in the review file (for checking parser misses) ====
    rv_ids = set(review["key_mlbam"].dropna().astype(str).str.split(", ").explode())
    rv_tx = tx.loc[tx["mlbam"].astype(str).isin(rv_ids) & tx["event_date"].dt.year.between(min(VALIDATE_SEASONS) - 1, max(VALIDATE_SEASONS))]
    rv_tx[["mlbam", "name", "tx_id", "event_date", "event", "retro_date", "type_code", "description"]].sort_values(
        ["name", "event_date"]).to_csv(DATA_DIR / "validation_review_transactions.csv", index=False)
    print(f"Transactions for these pitchers: data/validation_review_transactions.csv ({len(rv_tx):,} rows)")
else:
    print("Validation skipped (no VALIDATE_SEASONS or FanGraphs workbook not found).")

,season,FanGraphs pitcher rows,rebuilt pitcher rows,matched (IL within 3 days),linked by MLBAM only / FanGraphs id only,return date exact,return within 3 days,return missing in one source only
0,2021,687,689,95.9%,0 / 0,97.5%,99.6%,11
1,2022,559,563,91.4%,0 / 0,92.1%,96.8%,171
2,2023,502,497,92.8%,0 / 0,96.5%,97.9%,12


rows
only in   reason group                                            
FanGraphs FanGraphs id not in Chadwick and no rebuilt row...    11
          nearest rebuilt row N days off                        77
          pitcher has no rebuilt row this season                22
          rebuilt row already matched to another FanGraph...     2
rebuilt   nearest FanGraphs row N days off                      84
          no FanGraphs row for this pitcher this season         29

Every unmatched row, with its reason: data/validation_review.csv
Transactions for these pitchers: data/validation_review_transactions.csv (2,585 rows)


In [19]:
fg_only = review.loc[review["only in"] == "FanGraphs"]
print("FanGraphs-only rows:", len(fg_only))
print("  injury mentions COVID:", fg_only["injury"].astype(str).str.contains("covid", case=False).sum())
print("  of the rest, the reason:")
display(fg_only.loc[~fg_only["injury"].astype(str).str.contains("covid", case=False), "reason group"].value_counts())

reb = out.loc[(out["Pos"] == "P") & out["Season"].isin(VALIDATE_SEASONS)]
rd = pd.to_datetime(reb["Return Date"])
print("Rebuilt pitcher rows with a return date in Oct-Feb (off-season reinstatement):",
      int(rd.dt.month.isin([10, 11, 12, 1, 2]).sum()))
print("COVID-list transactions the parser skipped:",
      tx["description"].str.contains("COVID", case=False).sum())
display(review.groupby(["only in", "reason group"]).size().rename("rows").to_frame())

FanGraphs-only rows: 112
  injury mentions COVID: 15
  of the rest, the reason:


reason group
nearest rebuilt row N days off                                                              69
pitcher has no rebuilt row this season                                                      17
FanGraphs id not in Chadwick and no rebuilt row with that id                                 9
rebuilt row already matched to another FanGraphs row (FanGraphs lists this stint twice?)     2
Name: count, dtype: int64

Rebuilt pitcher rows with a return date in Oct-Feb (off-season reinstatement): 20
COVID-list transactions the parser skipped: 2


rows
only in   reason group                                            
FanGraphs FanGraphs id not in Chadwick and no rebuilt row...    11
          nearest rebuilt row N days off                        77
          pitcher has no rebuilt row this season                22
          rebuilt row already matched to another FanGraph...     2
rebuilt   nearest FanGraphs row N days off                      84
          no FanGraphs row for this pitcher this season         29

## 7. Write the combined workbook

The FanGraphs sheets are copied unchanged; one sheet per season in `BUILD_SEASONS` is added with the same
column names (plus `key_mlbam` and a few provenance columns). In the v4/v5 notebooks, point
`INJURY_WORKBOOK` at this file and add the new seasons to `SEASONS` / `TRAIN_SEASONS`.

In [20]:
with pd.ExcelWriter(OUTPUT_WORKBOOK) as xw:
    if FANGRAPHS_WORKBOOK.exists():
        src = pd.ExcelFile(FANGRAPHS_WORKBOOK)
        for sheet in src.sheet_names:
            pd.read_excel(src, sheet_name=sheet).to_excel(xw, sheet_name=str(sheet), index=False)
    for s in BUILD_SEASONS:
        sheet = out.loc[out["Season"] == s].drop(columns="Season")
        sheet.to_excel(xw, sheet_name=str(s), index=False)
summary = out.loc[out["Season"].isin(BUILD_SEASONS)].groupby("Season").agg(
    IL_rows=("Name", "size"), pitcher_rows=("Pos", lambda p: int((p == "P").sum())),
    pitchers=("key_mlbam", "nunique"), with_return_date=("Return Date", lambda r: int(r.notna().sum())))
display(summary)
print("Wrote", OUTPUT_WORKBOOK)

,IL_rows,pitcher_rows,pitchers,with_return_date
Season,,,,
2015,517,288,441,363
2016,553,302,471,381
2017,697,383,542,525
2018,740,432,584,565
2019,731,400,576,560


Wrote /workspaces/Sports-Analytics-and-Intelligence-Lab/MLB Injuries/MLB Injuries Raw Data 2015-2026.xlsx


## 8. Notes before using these seasons in v4 / v5

* **Check the validation table first.** Match rates in the high 90s on 2021–2023 mean the rebuilt rows can
  stand in for FanGraphs. Read the two unmatched files for any systematic miss before trusting 2015–2019.
* **No separate injury date.** MLB transactions give the IL date, not the date of injury or surgery, except
  that carry-overs get the original IL date. FanGraphs sometimes records an earlier injury/surgery date, so
  baselines for 2015–2019 may run a little closer to the injury. The 30-day-lagged baseline makes this moot.
* **2015 baselines and PITCHf/x.** The 365-day baseline for 2015 injuries reaches into 2014, which comes from
  PITCHf/x, not Statcast; its velocities are not directly comparable (measured at a different point).
  Either start the extended cohort in 2016 or restrict 2015 baselines to 2015 games.
* **Injury descriptions** come from the same text, so the arm / elbow patterns in v4/v5 apply as they are;
  check `injury_classification_review.csv` there again after adding these seasons.
* **2020** stays out (shortened season), as before.